## WAP to predict real estate price using decision trees.

In [2]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [3]:
spark = SparkSession.builder.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/04/03 21:17:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/04/03 21:17:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [8]:
data = spark.read.csv('datasets/real_estate.csv', inferSchema=True, header=True)

In [6]:
data.printSchema()
data.show(5)

root
 |-- id: long (nullable = true)
 |-- date: string (nullable = true)
 |-- price: double (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- sqft_living: integer (nullable = true)
 |-- sqft_lot: integer (nullable = true)
 |-- floors: double (nullable = true)
 |-- waterfront: integer (nullable = true)
 |-- view: integer (nullable = true)
 |-- condition: integer (nullable = true)
 |-- grade: integer (nullable = true)
 |-- sqft_above: integer (nullable = true)
 |-- sqft_basement: integer (nullable = true)
 |-- yr_built: integer (nullable = true)
 |-- yr_renovated: integer (nullable = true)
 |-- zipcode: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- sqft_living15: integer (nullable = true)
 |-- sqft_lot15: integer (nullable = true)

+----------+---------------+--------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------

In [11]:
features = data.columns
features.remove('date')
features.remove('price')

In [12]:
assembler = VectorAssembler(inputCols=features, outputCol='features')
evaluator = RegressionEvaluator(predictionCol='prediction', labelCol='price', metricName='rmse')

In [13]:
data = assembler.transform(data)
train_data, test_data = data.randomSplit([0.7, 0.3], seed=100)

In [15]:
dt = DecisionTreeRegressor(featuresCol='features', labelCol='price', predictionCol='prediction')
dt_model = dt.fit(train_data)
dt_pred = dt_model.transform(test_data)

In [16]:
evaluator.evaluate(dt_pred)

196814.41927703007